# BIOT 6900 · Module 7 · Real Disease Knowledge Graph (Neo4j)

### *The worked example: Alzheimer's in a real property-graph database*

The toy notebook built a graph by hand in NetworkX to **see** the ideas. This
one does it for real: a richer Alzheimer's dataset loaded into **Neo4j**,
queried with **Cypher**, and analysed for **biomarker prioritization** and
**patient stratification**. It is close in shape to your graded assignment, so
treat it as the template you will adapt to your own Part I disease.

**What's different from the toy**

- A real database (Neo4j) instead of an in-memory graph.
- A larger, multi-omics dataset that carries your Modules 2–6 evidence (GWAS p, RNA/protein, druggability) on the nodes, plus clinical phenotypes, pathways, drugs, variants, and a (synthetic) patient cohort.
- Queries written in **Cypher** (run in Neo4j); the two graph algorithms (PageRank, community) run client-side in **NetworkX**, so it works on any Neo4j tier including AuraDB Free.
- An optional **text-to-Cypher** helper (GenAI querying).

**Setup once:** a free Neo4j AuraDB instance (no install, no credit card) or a
local Community Edition. Nothing here needs paid software.


## 0 · Install packages and connect

First install the libraries into this notebook's kernel, then paste your Neo4j
connection details. For **AuraDB Free**: create a free instance at
`console.neo4j.io`, download the credentials, and copy the connection URI and
password. **Never commit your password** — keep it out of the notebook you push
(use an environment variable or a `.env` you git-ignore).

In [ ]:
# Install into THIS notebook's kernel (the leading % is required in Jupyter).
# The kernel may restart after installing — if it does, just run the cells
# again from the top. Installing is a one-time step per environment.
%pip install neo4j networkx matplotlib pandas requests

In [ ]:
from neo4j import GraphDatabase

# ---- fill these in from your Aura "Download credentials" file ----
# On AuraDB Free the three values come straight from that file:
#   NEO4J_URI       -> the neo4j+s:// connection URI
#   NEO4J_USERNAME  -> this is your INSTANCE ID (e.g. 4e779ede), NOT "neo4j"
#   NEO4J_PASSWORD  -> the generated password (paste it alone, nothing appended)
NEO4J_URI  = "neo4j+s://YOUR-INSTANCE.databases.neo4j.io"
NEO4J_USER = "YOUR-INSTANCE-ID"      # same id as in the URI; not the word "neo4j"
NEO4J_PASSWORD = "YOUR-PASSWORD"     # do not commit this — keep it in a git-ignored .env

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def run_query(cypher, **params):
    """Run Cypher and return rows as a list of dicts."""
    with driver.session() as session:
        return [r.data() for r in session.run(cypher, **params)]

try:
    driver.verify_connectivity()
    print("Connected to Neo4j.")
except Exception as e:
    print("Not connected yet — fill in your URI/password above. Details:", e)

## 1 · The data

A real Alzheimer's slice, assembled and **simplified for teaching** from public
sources (Open Targets, GWAS Catalog, HPO, Reactome, ChEMBL). The evidence on
each gene mirrors what you produced earlier in the course. The patient cohort is
**synthetic** — not real individuals.

In [ ]:
import pandas as pd
import networkx as nx

# ---------------------------------------------------------------------------
# A real Alzheimer's slice, assembled and SIMPLIFIED for teaching from public
# sources (Open Targets, GWAS Catalog, HPO, Reactome, ChEMBL). Numbers are
# representative / rounded and illustrative, not exact study values. For real
# work, pull live data (see the optional Open Targets cell below).
# ---------------------------------------------------------------------------

# genes: symbol, ensembl, gwas_p, rna_lfc, prot_lfc, druggability, role
GENES = [
    ("APOE",  "ENSG00000130203", 1e-80, 0.9, 1.1, 0.35, "risk"),
    ("TREM2", "ENSG00000095970", 1e-25, 1.4, 1.0, 0.55, "risk"),
    ("BIN1",  "ENSG00000136717", 1e-30, 0.7, 0.5, 0.30, "risk"),
    ("CLU",   "ENSG00000120885", 1e-20, 0.6, 0.7, 0.25, "risk"),
    ("PICALM","ENSG00000073921", 1e-18, 0.4, 0.3, 0.20, "risk"),
    ("ABCA7", "ENSG00000064687", 1e-15, 0.5, 0.4, 0.45, "risk"),
    ("SORL1", "ENSG00000137642", 1e-14, 0.8, 0.6, 0.40, "risk"),
    ("CR1",   "ENSG00000203710", 1e-12, 0.3, 0.2, 0.30, "risk"),
    ("CD33",  "ENSG00000105383", 1e-11, 0.9, 0.7, 0.50, "risk"),
    ("MS4A4A","ENSG00000110077", 1e-10, 0.6, 0.4, 0.35, "risk"),
    ("APP",   "ENSG00000142192", 1e-9,  0.5, 0.8, 0.60, "causal"),
    ("PSEN1", "ENSG00000080815", 1e-8,  0.3, 0.4, 0.55, "causal"),
    ("ACHE",  "ENSG00000087085", None,  0.1, 0.1, 0.90, "symptomatic target"),
    ("GRIN2A","ENSG00000183454", None,  0.2, 0.1, 0.80, "symptomatic target"),
    # related-disease genes (enable the repurposing walk)
    ("LRRK2", "ENSG00000188906", 1e-15, 0.4, 0.3, 0.70, "Parkinson's risk"),
    ("MAPT",  "ENSG00000186868", 1e-12, 0.6, 0.9, 0.50, "FTD / tau"),
]

DISEASES = ["Alzheimer's disease", "Parkinson's disease", "Frontotemporal dementia"]

PHENOTYPES = ["Memory impairment", "Cognitive decline", "Neuroinflammation",
              "Amyloid plaques", "Neurofibrillary tangles"]

PATHWAYS = ["Lipid transport", "Amyloid processing", "Microglial response",
            "Endocytosis", "Cholinergic signaling", "Glutamatergic signaling"]

# drug -> target gene, status
DRUGS = [
    ("Donepezil",   "ACHE",   "approved"),
    ("Galantamine", "ACHE",   "approved"),
    ("Rivastigmine","ACHE",   "approved"),
    ("Memantine",   "GRIN2A", "approved"),
    ("Lecanemab",   "APP",    "approved"),
    ("Aducanumab",  "APP",    "approved"),
    ("LRRK2 inhibitor (investigational)", "LRRK2", "investigational"),
    ("Anti-tau (investigational)",        "MAPT",  "investigational"),
]

VARIANTS = [("rs429358", "APOE"), ("rs75932628", "TREM2")]

# synthetic patient cohort (NOT real individuals)
PATIENTS = [
    ("P01", 72, "e3/e4", 0.88, "case"), ("P02", 69, "e4/e4", 0.95, "case"),
    ("P03", 75, "e3/e3", 0.40, "case"), ("P04", 81, "e3/e4", 0.78, "case"),
    ("P05", 66, "e2/e3", 0.22, "control"), ("P06", 70, "e3/e3", 0.35, "control"),
    ("P07", 77, "e3/e4", 0.70, "case"), ("P08", 68, "e2/e3", 0.18, "control"),
    ("P09", 74, "e4/e4", 0.92, "case"), ("P10", 63, "e3/e3", 0.30, "control"),
    ("P11", 79, "e3/e4", 0.66, "case"), ("P12", 71, "e3/e3", 0.44, "control"),
]

AD = "Alzheimer's disease"

# ----- build node and edge collections ------------------------------------
NODES = {}   # name -> dict(kind=..., **props)
EDGES = []   # list of dict(src, rel, dst, **props)

def add_node(name, kind, **props): NODES[name] = dict(kind=kind, **props)
def add_edge(src, rel, dst, **props): EDGES.append(dict(src=src, rel=rel, dst=dst, **props))

for sym, ens, p, rna, prot, drug, role in GENES:
    add_node(sym, "gene", ensembl=ens, gwas_p=p, rna_lfc=rna, prot_lfc=prot,
             druggability=drug, role=role)
for d in DISEASES: add_node(d, "disease")
for ph in PHENOTYPES: add_node(ph, "phenotype")
for pw in PATHWAYS: add_node(pw, "pathway")
for name, tgt, status in DRUGS: add_node(name, "drug", status=status)
for v, g in VARIANTS: add_node(v, "variant")
for pid, age, geno, prs, status in PATIENTS:
    add_node(pid, "patient", age=age, apoe_genotype=geno, prs_percentile=prs, status=status)

# gene -ASSOCIATED_WITH-> disease (Alzheimer's)
AD_GENES = ["APOE","TREM2","BIN1","CLU","PICALM","ABCA7","SORL1","CR1","CD33",
            "MS4A4A","APP","PSEN1","ACHE","GRIN2A"]
for g in AD_GENES:
    p = NODES[g].get("gwas_p")
    add_edge(g, "ASSOCIATED_WITH", AD, pvalue=p, source=("GWAS" if p else "pharmacology"))
add_edge("LRRK2", "ASSOCIATED_WITH", "Parkinson's disease", pvalue=1e-15, source="GWAS")
add_edge("MAPT",  "ASSOCIATED_WITH", "Frontotemporal dementia", pvalue=1e-12, source="GWAS")

# disease -HAS_PHENOTYPE-> phenotype
DIS_PHENO = {
    AD: ["Memory impairment","Cognitive decline","Neuroinflammation","Amyloid plaques","Neurofibrillary tangles"],
    "Parkinson's disease": ["Cognitive decline","Neuroinflammation"],
    "Frontotemporal dementia": ["Memory impairment","Cognitive decline"],
}
for d, phs in DIS_PHENO.items():
    for ph in phs: add_edge(d, "HAS_PHENOTYPE", ph)

# gene -PARTICIPATES_IN-> pathway
GENE_PATH = {
    "Lipid transport": ["APOE","CLU","ABCA7","SORL1"],
    "Amyloid processing": ["APP","PSEN1","BIN1"],
    "Microglial response": ["TREM2","CR1","CD33","MS4A4A","ABCA7"],
    "Endocytosis": ["BIN1","PICALM","SORL1"],
    "Cholinergic signaling": ["ACHE"],
    "Glutamatergic signaling": ["GRIN2A"],
}
for pw, genes in GENE_PATH.items():
    for g in genes: add_edge(g, "PARTICIPATES_IN", pw)

# drug -TARGETS-> gene
for name, tgt, status in DRUGS: add_edge(name, "TARGETS", tgt)

# variant -OCCURS_IN-> gene
for v, g in VARIANTS: add_edge(v, "OCCURS_IN", g)

# patient -CARRIES-> rs429358 (APOE e4 carriers)
for pid, age, geno, prs, status in PATIENTS:
    if "e4" in geno: add_edge(pid, "CARRIES", "rs429358")

genes_df = pd.DataFrame(GENES, columns=["symbol","ensembl","gwas_p","rna_lfc",
                                        "prot_lfc","druggability","role"])
print("nodes:", len(NODES), " edges:", len(EDGES))
print("by kind:", pd.Series([d["kind"] for d in NODES.values()]).value_counts().to_dict())


In [ ]:
genes_df   # the multi-omics evidence that will sit on the gene nodes

#### Optional — pull live from Open Targets (the real-API pattern)

For real work you fetch associations from an API rather than hand-curating them.
This cell shows the shape using Open Targets' public GraphQL endpoint (no key).
It is **optional** and needs network access; field names evolve, so check the
schema at `platform.opentargets.org` if it changes.

In [ ]:
# OPTIONAL — needs: pip install requests, and network access
import requests

def open_targets_associations(efo_id="MONDO_0004975", n=20):   # Alzheimer's disease
    q = """
    query assoc($efo: String!, $n: Int!) {
      disease(efoId: $efo) {
        name
        associatedTargets(page: {index: 0, size: $n}) {
          rows { target { approvedSymbol } score }
        }
      }
    }"""
    r = requests.post("https://api.platform.opentargets.org/api/v4/graphql",
                      json={"query": q, "variables": {"efo": efo_id, "n": n}}, timeout=30)
    data = r.json()["data"]["disease"]
    return [(row["target"]["approvedSymbol"], round(row["score"], 3))
            for row in data["associatedTargets"]["rows"]]

# print(open_targets_associations())   # uncomment to run

## 2 · Load the property graph into Neo4j

Every node gets its specific label (`:Gene`, `:Disease`, …) **and** a shared
`:Entity` label with a unique `name`, so edges can be matched by name and the
whole graph round-trips cleanly. We load in batches with `UNWIND` — the standard
pattern for loading many rows in one query.

In [ ]:
# uniqueness constraint (also creates an index on :Entity(name))
run_query("CREATE CONSTRAINT entity_name IF NOT EXISTS "
          "FOR (n:Entity) REQUIRE n.name IS UNIQUE")

KIND_LABEL = {"gene":"Gene","disease":"Disease","phenotype":"Phenotype",
              "pathway":"Pathway","drug":"Drug","variant":"Variant","patient":"Patient"}

# nodes, grouped by kind so each gets the right label
for kind, label in KIND_LABEL.items():
    rows = [dict(name=n, **{k: v for k, v in d.items() if k != "kind"})
            for n, d in NODES.items() if d["kind"] == kind]
    if not rows:
        continue
    run_query(f"""
        UNWIND $rows AS row
        MERGE (e:{label}:Entity {{name: row.name}})
        SET e += row
    """, rows=rows)

print("nodes loaded")

In [ ]:
# edges, grouped by relationship type (rel type can't be parameterised directly)
from collections import defaultdict
by_rel = defaultdict(list)
for e in EDGES:
    props = {k: v for k, v in e.items() if k not in ("src","dst","rel")}
    by_rel[e["rel"]].append({"src": e["src"], "dst": e["dst"], "props": props})

for rel, rows in by_rel.items():
    run_query(f"""
        UNWIND $rows AS row
        MATCH (a:Entity {{name: row.src}})
        MATCH (b:Entity {{name: row.dst}})
        MERGE (a)-[r:{rel}]->(b)
        SET r += row.props
    """, rows=rows)

counts = run_query("""
    MATCH (n) RETURN labels(n)[0] AS label, count(*) AS n ORDER BY n DESC
""")
print("node counts:", counts)
print("relationships:", run_query("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

## 3 · A note on identifiers

Here the data is already keyed on one gene symbol, so the graph connects. On
real data the same gene arrives as a symbol, an Ensembl id, a UniProt accession
and an Entrez id; if those are not resolved to **one** node, edges silently
never form. Map everything through a reference (HGNC, MyGene.info) before
loading. This is where most real knowledge-graph work lives.

## 4 · Ask the opening question, in Cypher

The question from the first lecture slide, now against a real database.

In [ ]:
# 2 hops: drugs acting on genes associated with Alzheimer's
run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(dis:Disease)
    WHERE dis.name = $disease
    RETURN d.name AS drug, g.name AS gene, d.status AS status
    ORDER BY drug
""", disease="Alzheimer's disease")

In [ ]:
# 5 hops: drugs on genes of diseases that SHARE A SYMPTOM with Alzheimer's
run_query("""
    MATCH (d:Drug)-[:TARGETS]->(g:Gene)-[:ASSOCIATED_WITH]->(other:Disease)
          -[:HAS_PHENOTYPE]->(p:Phenotype)<-[:HAS_PHENOTYPE]-(ad:Disease)
    WHERE ad.name = $disease AND other <> ad
    RETURN d.name AS repurposing_candidate, g.name AS gene,
           other.name AS via_disease, collect(DISTINCT p.name) AS shared_symptoms
    ORDER BY repurposing_candidate
""", disease="Alzheimer's disease")

## 5 · Biomarker prioritization (centrality)

We compute PageRank in **Python (NetworkX)** on the graph you loaded. Neo4j stays
your store and query engine — every Cypher cell above runs there — but the two
graph *algorithms* (PageRank here, community detection next) run client-side, so
this works on **any** Neo4j tier, including AuraDB Free. For graphs far larger
than a disease model you'd move these into a GDS Session (see §9); at this
course's scale, NetworkX is the simpler, portable choice.

We fuse the PageRank score with the Modules 2–6 evidence on each gene: a gene
that is both a **hub** and **well-supported** and **druggable** is a strong
biomarker/target candidate — the Part I → Part III thread of the course. This
cell also defines `G`, used by the visualization in §7.

In [ ]:
import pandas as pd
import networkx as nx

# Build a NetworkX graph from the same data you loaded into Neo4j. Neo4j stays
# your store and query engine (every Cypher cell runs there); the two graph
# ALGORITHMS run client-side so this works on any Neo4j tier, AuraDB Free included.
G = nx.DiGraph()
for name, d in NODES.items():
    G.add_node(name, **d)
for e in EDGES:
    G.add_edge(e["src"], e["dst"], **{k: v for k, v in e.items() if k not in ("src", "dst")})

pr = nx.pagerank(G)
rows = [{"gene": n, "pagerank": round(pr[n], 4), "gwas_p": d.get("gwas_p"),
         "druggability": d.get("druggability"), "role": d.get("role")}
        for n, d in G.nodes(data=True) if d["kind"] == "gene"]
biomarkers = pd.DataFrame(rows).sort_values("pagerank", ascending=False)
print("Genes by PageRank, fused with your Part I evidence:")
print(biomarkers.to_string(index=False))


## 6 · Communities and patient stratification

**Community detection** groups densely connected nodes — pathway/disease modules
here, patient groups in a larger graph. We use modularity-based clustering
(Louvain-style) in NetworkX on the same `G`.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

comms = greedy_modularity_communities(G.to_undirected())
print(f"{len(comms)} communities:")
for i, c in enumerate(comms):
    print(f"  {i}: {sorted(c)}")


**Patient stratification** over the (synthetic) cohort: how do APOE-genotype
groups differ in case rate and polygenic risk? This one runs in Cypher, back in
Neo4j. It is the "patient characteristics" layer the assignment asks for.

In [ ]:
run_query("""
    MATCH (p:Patient)
    RETURN p.apoe_genotype AS genotype,
           count(*) AS n,
           sum(CASE WHEN p.status = 'case' THEN 1 ELSE 0 END) AS cases,
           round(avg(p.prs_percentile), 2) AS mean_prs
    ORDER BY mean_prs DESC
""")

In [ ]:
# who carries the APOE risk allele (variant-level link)?
run_query("""
    MATCH (p:Patient)-[:CARRIES]->(v:Variant)-[:OCCURS_IN]->(g:Gene)
    RETURN v.name AS variant, g.name AS gene,
           collect(p.name) AS carriers, count(*) AS n_carriers
""")

## 7 · Visualize

Neo4j Browser (at your AuraDB console) is the native way to explore the graph
interactively. For a figure in the notebook, pull a subgraph into NetworkX and
draw it — reuse the `draw_graph` helper from the toy notebook, colouring by
`kind`.

In [ ]:
import matplotlib.pyplot as plt

KIND_COLOR = {"gene":"#D41B2C","disease":"#3E6F99","phenotype":"#2FA3D6",
              "pathway":"#00A58C","drug":"#9A5FC0","variant":"#FF854F","patient":"#8A94A3"}

# a readable subgraph: drop patients for the overview
H = G.subgraph([n for n, d in G.nodes(data=True) if d["kind"] != "patient"])
pos = nx.spring_layout(H, seed=7, k=0.6)
plt.figure(figsize=(13, 8))
nx.draw_networkx_edges(H, pos, edge_color="#C2CCD6", width=1.0, arrows=False)
nx.draw_networkx_nodes(H, pos, node_size=600,
                       node_color=[KIND_COLOR[H.nodes[n]["kind"]] for n in H.nodes()])
nx.draw_networkx_labels(H, pos, font_size=7)
plt.axis("off"); plt.title("Alzheimer's knowledge graph (patients hidden)"); plt.show()

## 8 · Optional — text-to-Cypher (GenAI querying)

A language model can turn a plain-English question into Cypher, so a biologist
need not hand-write graph syntax. This is **optional** and needs the `anthropic`
package and an API key. Its failure mode is a query that *runs* but answers a
subtly different question, so always read the generated Cypher before trusting
the result.

In [ ]:
# OPTIONAL — needs: pip install anthropic, and an API key.
# import anthropic
# client = anthropic.Anthropic(api_key="YOUR_KEY")
#
# SCHEMA = """
# Node labels: Gene, Disease, Phenotype, Pathway, Drug, Variant, Patient (all also :Entity {name}).
# Relationships:
#   (Gene)-[:ASSOCIATED_WITH {pvalue, source}]->(Disease)
#   (Disease)-[:HAS_PHENOTYPE]->(Phenotype)
#   (Gene)-[:PARTICIPATES_IN]->(Pathway)
#   (Drug)-[:TARGETS]->(Gene)
#   (Variant)-[:OCCURS_IN]->(Gene)
#   (Patient)-[:CARRIES]->(Variant)
# """
#
# def ask_graph(question, model="claude-sonnet-4-5"):   # use a current model id
#     msg = client.messages.create(
#         model=model, max_tokens=400,
#         system="Translate the question to a single Cypher query for this schema. "
#                "Return ONLY Cypher, no prose.\n" + SCHEMA,
#         messages=[{"role": "user", "content": question}])
#     cypher = msg.content[0].text.strip().strip("`")
#     print("Generated Cypher:\n", cypher, "\n")   # ALWAYS read it before trusting it
#     return run_query(cypher)
#
# ask_graph("Which pathways are shared by the top three GWAS genes for Alzheimer's?")
print("Optional cell — uncomment and add an API key to try text-to-Cypher.")

## 9 · Reset and reuse

To start over, clear the graph. Then, for the assignment, swap the dataset in
§1 for **your own Part I disease**: load your `targets_ad.csv` genes and their
evidence, add the phenotypes, pathways and at least one related disease, and
re-run every section.

```cypher
MATCH (n) DETACH DELETE n        // wipes the whole database — run deliberately
```

#### What you built

| Step | In this notebook |
|---|---|
| Real database | loaded a multi-omics Alzheimer's graph into Neo4j (§2) |
| Cypher queries | the 2-hop and 5-hop opening question (§4) |
| Biomarker prioritization | PageRank fused with GWAS/druggability evidence (§5) |
| Patient stratification | APOE-genotype cohort summary + risk-allele carriers (§6) |
| GenAI querying | optional text-to-Cypher (§8) |

**The assignment** is this, for your disease, with a 3–4 page interpretation.


## Going further — GDS Sessions (optional, for scale)

We ran PageRank and community detection in NetworkX because it works on every
Neo4j tier and our graphs are small. On **Aura Graph Analytics (serverless)**,
the production way to run these *inside* the platform is a **GDS Session**: you
install the `graphdatascience` Python client, authenticate with **Aura API
credentials** (a client id/secret you generate in the console, separate from the
database password), create a session with a memory allocation, project your
graph into it, then call `gds.pageRank` / `gds.louvain`. It pays off for graphs
far larger than a disease model, at the cost of extra setup. For this course's
scale, the NetworkX route above is the simpler choice and gives the same result.
